# Partition uncertainties in projections of impacts

1. Of just mean estimates
2. With regression uncertainties
3. Bootstrapped over number of runs (maybe spin that off into a 4b?)
4. TODO: Without MERRA2

For different sets of runs: 
- a "balanced panel" of GCMs / runs
- all available runs at GWL 2n

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from matplotlib import colors as mcolors

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc
from funcs_processing import calc_uncerts
from funcs_plot import plot_map, vals_to_color_triangle

dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44943 instead
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/contextlib.py:148: UserWarning: Creating scratch directories is taking a surprisingly long time. (2.86s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44943/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44943/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46773,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/44943/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46417,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/33693/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:43231,


2026-03-16 14:47:34,868 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7a255d54ab5b8ffd9af556eabcab41d8 initialized by task ('rechunk-merge-rechunk-transfer-297e3012464976c3787bf9e86e360945', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 162, 0) executed on worker tcp://127.0.0.1:44097
2026-03-16 14:50:35,322 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7a255d54ab5b8ffd9af556eabcab41d8 deactivated due to stimulus 'task-finished-1773694235.3213098'
2026-03-16 14:55:09,961 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ea425a4d42a5086538adeec4c620abcf initialized by task ('rechunk-merge-rechunk-transfer-de3dbb9142f0674fad5f3f9fb0cf8898', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0) executed on worker tcp://127.0.0.1:46417
2026-03-16 14:55:12,872 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ea425a4d42a5086538adeec4c620abcf deactivated due to stimulus 'task-finished-1773694512.8712294'
2026-03-16 14:55:22,644 - distributed.shuffle._sch

## Climate uncertainty partitioning

In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

In [4]:
proc = 'mortality'

#---------------------------
# Setup
#---------------------------
# Load impacts projections
fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*.zarr'))
def load_impact(fn):
    ds_tmp = xr.open_zarr(fn,consolidated=False)
    ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
    ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})

    ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
    ds_tmp = ds_tmp.reset_index('idv')
    ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
    return ds_tmp

with warnings.catch_warnings():
    # Silence zarr warnings related to the /.done flags used
    warnings.filterwarnings('ignore')
    dss = xr.concat([load_impact(fn) for fn in fns],
                dim='idv',join='outer')

In [24]:
overwrite = True
for proc in params_proc:
    output_fn = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}.zarr'
    if overwrite or (not os.path.exists(output_fn)):
        #---------------------------
        # Setup
        #---------------------------
        # Load impacts projections
        fns = np.sort(glob.glob(f'{dir_list['proc']}impact_projs/{proc}/d{proc}*.zarr'))
        def load_impact(fn):
            ds_tmp = xr.open_zarr(fn,consolidated=False)
            ds_tmp = ds_tmp.set_coords('locid') # Should be coordinate, instead of concatenated variable
            ds_tmp = ds_tmp.assign_coords({'model':(('idv'),[re.split(r'\_',re.split(r'\/',fn)[-1])[2]]*ds_tmp.sizes['idv'])})
        
            ds_tmp['exprun'] = ds_tmp['experiment']+ds_tmp['run']
            ds_tmp = ds_tmp.reset_index('idv')
            ds_tmp = ds_tmp.set_index(idv = ['model','exprun'])
            return ds_tmp
        
        with warnings.catch_warnings():
            # Silence zarr warnings related to the /.done flags used
            warnings.filterwarnings('ignore')
            dss = xr.concat([load_impact(fn) for fn in fns],
                        dim='idv',join='outer')
        
        
        dss = dss.chunk({'poly_idx':50,'idv':-1,'proj_base':-1,'gwl':-1})

        # Save temporary rechunked file to make dask graphing more stable
        tmp_fn = dir_list['proc']+'tmp/uncert_rechunk_cache_tmp.zarr'
        utility_save(dss.reset_index('idv').drop_encoding(),tmp_fn,zarr_mode = 'w')
        dss = xr.open_zarr(tmp_fn).set_index(idv = ['model','exprun'])
        
        # Figure out how many BCD-ME members reach a given GWL
        gwl_counter = dss.maxgwl.load().reset_index('idv').groupby(['model','maxgwl']).count()
        gwl_counter = gwl_counter.sel(maxgwl = gwl_counter.maxgwl.values[::-1]).cumsum('maxgwl')
        #print(gwl_counter)
        
        uncerts_all = []
        #---------------------------
        # Uncertainty partitioning over all runs
        #---------------------------
        dss_tmp = dss.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        # Calculate uncertainty partitioning
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':['allruns']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Uncertainty partitioning over a balanced panel up to GWL 3
        #---------------------------
        max_gwl = 3
        nruns = 13
        
        gwlmods = gwl_counter.sel(maxgwl=3)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.isel(idv = dss.maxgwl>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
        dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
        dss_tmp = dss_tmp.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods']})
        
        uncerts_all.append(uncerts)

        #---------------------------
        # Uncertainty partitioning over a balanced panel up to GWL 3
        # without MERRA2
        #---------------------------
        max_gwl = 3
        nruns = 13
        
        gwlmods = gwl_counter.sel(maxgwl=3)
        proc_mods = gwlmods.isel(model = gwlmods>=nruns).model.values
        
        dss_tmp = dss.isel(idv = dss.maxgwl>=max_gwl).groupby('model').map(lambda x: x.isel(idv=slice(0, nruns)))
        dss_tmp = dss_tmp.sel(proj_base = [pb for pb in dss_tmp.proj_base.values if pb != 'MERRA2'])
        dss_tmp = dss_tmp.isel(idv=[m in proc_mods for m in dss_tmp.model.values]).chunk({'idv':-1})
        dss_tmp = dss_tmp.unstack('idv')['d'+proc]
        dss_tmp = dss_tmp.chunk({'model':-1,'exprun':-1})
        
        with warnings.catch_warnings():
            # Ignoring chunk increase warnings and divide by 0s in uncertainty 
            # partitioning (happens over ocean, e.g.)
            warnings.filterwarnings('ignore')
            uncerts = xr.map_blocks(calc_uncerts,dss_tmp).compute()
        
        uncerts = uncerts.expand_dims({'sample':[f'{str(nruns)}runs{str(dss_tmp.sizes['model'])}mods_noMERRA2']})
        
        uncerts_all.append(uncerts)
        
        #---------------------------
        # Export
        #---------------------------
        # Concatenate into single ds
        uncerts_all = xr.concat(uncerts_all,dim='sample')
        uncerts_all = uncerts_all.where(uncerts_all.sum('typ')>0)
        
        # Export
        uncerts_all.attrs = {'SOURCE':'4_partition_uncertainty.ipynb',
                             'DESCRIPTION':f'Uncertainty partitioning of projections of {proc} between model, internal, and reanalysis sources'}
        
        utility_save(uncerts_all,output_fn,zarr_mode = 'w')

        os.system('rm -rf '+tmp_fn)
        print('Temporary file '+tmp_fn+' removed.')
    else:
        print(output_fn+' exists!')
    


/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change with

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_mortality.zarr removed to allow overwrite!


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:242: RuntimeWarning: Degrees of freedom <= 0 for slice.
  result = getattr(npmodule, name)(values, axis=axis, **kwargs)
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/nputils.py:

/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_payroll.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/uncert_parts/uncert_parts_payroll.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/uncert_rechunk_cache_tmp.zarr removed.


In [8]:
dss_tmp

<xarray.DataArray 'dmortality' (resolution: 2, timeframe: 2, obstreat: 3,
                                spec: 1, indflag: 2, proj_base: 4,
                                poly_idx: 24378, gwl: 8, model: 12, exprun: 159)> Size: 286GB
dask.array<rechunk-merge, shape=(2, 2, 3, 1, 2, 4, 24378, 8, 12, 159), dtype=float64, chunksize=(2, 2, 3, 1, 2, 4, 500, 8, 12, 159), chunktype=numpy.ndarray>
Coordinates: (12/14)
  * resolution  (resolution) object 16B '025deg' '1deg'
  * timeframe   (timeframe) object 16B '1980-2010' 'ALLYEARS'
  * obstreat    (obstreat) object 24B 'joint' 'jointnotfes' 'sep'
  * spec        (spec) object 8B 'pooled'
  * indflag     (indflag) object 16B 'noInd' 'wInd'
  * proj_base   (proj_base) <U6 96B 'ERA5' 'GMFD' 'JRA-3Q' 'MERRA2'
    ...          ...
  * gwl         (gwl) float64 64B 0.61 1.0 1.5 2.0 2.5 3.0 3.5 4.0
  * model       (model) <U13 624B 'ACCESS-ESM1-5' 'CESM2-WACCM' ... 'MRI-ESM2-0'
  * exprun      (exprun) <U20 13kB 'ssp119r1i1p1f2' ... 'ssp585r9i1p2f1'
    experiment  (model, exprun) object 15kB dask.array<chunksize=(12, 159), meta=np.ndarray>
    maxgwl      (model, exprun) float64 15kB dask.array<chunksize=(12, 159), meta=np.ndarray>
    run         (model, exprun) object 15kB dask.array<chunksize=(12, 159), meta=np.ndarray>

## Climate and regression uncertainty partitioning